## yolo model starts 

In [26]:
import os
import sys
from pathlib import Path

# Find project root (folder that contains "src")
p = Path().resolve()
project_root = None
for parent in [p] + list(p.parents):
    if (parent / "src").is_dir():
        project_root = parent
        break

if project_root is None:
    raise FileNotFoundError("Could not find project root containing 'src' folder")

# Set working directory and PYTHONPATH
os.chdir(project_root)
sys.path.insert(0, str(project_root))

print("Working directory:", Path().resolve())
print("src folder found:", (Path().resolve() / "src").is_dir())

# Core imports
import cv2
import numpy as np
from tqdm import tqdm
import yaml


Working directory: E:\dl_cv_mini_project\road_lane_detection
src folder found: True


In [27]:
# YOLO + dataset imports
from ultralytics import YOLO
from src.dataset import TuSimpleSegmentationDataset

print("YOLO and TuSimpleSegmentationDataset imported successfully.")


YOLO and TuSimpleSegmentationDataset imported successfully.


In [28]:
from pathlib import Path

# New YOLO-only dataset folder (isolated)
YOLO_ROOT = Path("data/yolo_lane_seg_nb")

YOLO_IMG_TRAIN = YOLO_ROOT / "images/train"
YOLO_IMG_VAL   = YOLO_ROOT / "images/val"
YOLO_LAB_TRAIN = YOLO_ROOT / "labels/train"
YOLO_LAB_VAL   = YOLO_ROOT / "labels/val"

YOLO_YAML_PATH = YOLO_ROOT / "lane.yaml"

# Output folders (YOLO only)
YOLO_PRED_DIR = Path("outputs/yolo_predictions")
YOLO_CMP_DIR  = Path("outputs/compare_unet_vs_yolo")

# Create all folders
for d in [
    YOLO_IMG_TRAIN, YOLO_IMG_VAL,
    YOLO_LAB_TRAIN, YOLO_LAB_VAL,
    YOLO_PRED_DIR, YOLO_CMP_DIR
]:
    d.mkdir(parents=True, exist_ok=True)

print("YOLO dataset root:", YOLO_ROOT.resolve())
print("YOLO predictions dir:", YOLO_PRED_DIR.resolve())
print("Comparison dir:", YOLO_CMP_DIR.resolve())


YOLO dataset root: E:\dl_cv_mini_project\road_lane_detection\data\yolo_lane_seg_nb
YOLO predictions dir: E:\dl_cv_mini_project\road_lane_detection\outputs\yolo_predictions
Comparison dir: E:\dl_cv_mini_project\road_lane_detection\outputs\compare_unet_vs_yolo


In [29]:
def mask_to_polygons(mask_u8):
    """
    Better polygon conversion for thin lane masks:
    - Dilate slightly so lanes become learnable polygons
    - Preserve more contour detail
    """
    if mask_u8.max() <= 1:
        mask_u8 = (mask_u8 * 255).astype(np.uint8)
    else:
        mask_u8 = mask_u8.astype(np.uint8)

    # Thicken lanes slightly (critical for YOLO)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    mask_u8 = cv2.dilate(mask_u8, kernel, iterations=1)

    contours, _ = cv2.findContours(mask_u8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)

    polys = []
    for c in contours:
        if cv2.contourArea(c) < 200:
            continue
        # Less aggressive simplification than before
        c2 = cv2.approxPolyDP(c, epsilon=1.0, closed=True)
        if len(c2) < 10:
            continue
        polys.append(c2.reshape(-1, 2))

    return polys



def write_yolo_seg_label(txt_path, polys, W, H, cls_id=0):
    """
    YOLO-seg label format:
    class x1 y1 x2 y2 ... (normalized)
    """
    lines = []
    for pts in polys:
        coords = []
        for x, y in pts:
            coords.append(f"{x / W:.6f}")
            coords.append(f"{y / H:.6f}")
        if len(coords) >= 6:
            lines.append(f"{cls_id} " + " ".join(coords))
    txt_path.write_text("\n".join(lines), encoding="utf-8")


def export_split_to_yolo(split):
    ds = TuSimpleSegmentationDataset(split=split)

    if split == "train":
        img_dir, lab_dir = YOLO_IMG_TRAIN, YOLO_LAB_TRAIN
    else:
        img_dir, lab_dir = YOLO_IMG_VAL, YOLO_LAB_VAL

    img_dir.mkdir(parents=True, exist_ok=True)
    lab_dir.mkdir(parents=True, exist_ok=True)

    for i in tqdm(range(len(ds)), desc=f"Exporting {split}"):
        x, y = ds[i]

        img = (x.numpy().transpose(1, 2, 0) * 255).clip(0, 255).astype(np.uint8)
        mask = (y.squeeze().numpy() > 0.5).astype(np.uint8)

        H, W = mask.shape
        polys = mask_to_polygons(mask)

        # save image
        img_path = img_dir / f"{i:06d}.jpg"
        cv2.imwrite(str(img_path), cv2.cvtColor(img, cv2.COLOR_RGB2BGR))

        # save label
        lab_path = lab_dir / f"{i:06d}.txt"
        write_yolo_seg_label(lab_path, polys, W, H, cls_id=0)


# ---- RUN EXPORT ----
export_split_to_yolo("train")
export_split_to_yolo("val")

print("YOLO dataset export complete.")
print("Images:", YOLO_ROOT / "images")
print("Labels:", YOLO_ROOT / "labels")


Exporting train:   0%|          | 0/3264 [00:00<?, ?it/s]

Exporting val: 100%|██████████| 362/362 [00:04<00:00, 82.85it/s]

YOLO dataset export complete.
Images: data\yolo_lane_seg_nb\images
Labels: data\yolo_lane_seg_nb\labels


In [30]:
# Create YOLO dataset YAML for segmentation
data_yaml = {
    "path": str(YOLO_ROOT.as_posix()),
    "train": "images/train",
    "val": "images/val",
    "names": {
        0: "lane"
    }
}

YOLO_ROOT.mkdir(parents=True, exist_ok=True)

with open(YOLO_YAML_PATH, "w", encoding="utf-8") as f:
    yaml.safe_dump(data_yaml, f, sort_keys=False)

print("YOLO YAML created at:", YOLO_YAML_PATH.resolve())
print("\n--- lane.yaml content ---")
print(open(YOLO_YAML_PATH, "r", encoding="utf-8").read())


YOLO YAML created at: E:\dl_cv_mini_project\road_lane_detection\data\yolo_lane_seg_nb\lane.yaml

--- lane.yaml content ---
path: data/yolo_lane_seg_nb
train: images/train
val: images/val
names:
  0: lane



In [31]:
from ultralytics import YOLO
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# Stronger segmentation model
model = YOLO("yolo11s-seg.pt")

train_results = model.train(
    data=str(YOLO_YAML_PATH),
    epochs=50,        # 10 is too low for lanes
    imgsz=960,        # higher resolution helps thin structures
    device=0,         # GPU
    batch=8,          # reduce to 4 if OOM
    workers=0,        # Windows-safe
    cos_lr=True,
    patience=20
)

print("Training complete.")


CUDA available: True
GPU: NVIDIA GeForce RTX 4060 Laptop GPU
Ultralytics 8.3.247  Python-3.11.9 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=data\yolo_lane_seg_nb\lane.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=960, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train2, nbs=64, nms=False, opset=None, optimize=Fals

In [35]:
from ultralytics import YOLO
from pathlib import Path

BEST_PT = Path("runs/segment/train2/weights/best.pt")
assert BEST_PT.exists(), f"Missing: {BEST_PT.resolve()}"

yolo = YOLO(str(BEST_PT))
print("Loaded:", BEST_PT.resolve())


Loaded: E:\dl_cv_mini_project\road_lane_detection\runs\segment\train2\weights\best.pt


In [36]:
from pathlib import Path

selected_txt = Path("outputs/random_test_results/selected_files.txt")
assert selected_txt.exists(), f"Missing: {selected_txt.resolve()}"

picks = [x.strip() for x in selected_txt.read_text(encoding="utf-8").splitlines() if x.strip()]
print("Images to predict:", len(picks))
print("Example:", picks[0])


Images to predict: 20
Example: E:\dl_cv_mini_project\road_lane_detection\data\raw\tusimple\test_set\clips\0530\1492628131090172797_0\11.jpg


In [37]:
import cv2
from pathlib import Path

YOLO_OUT = Path("outputs/yolo_predictions_same20_no_boxes")
YOLO_OUT.mkdir(parents=True, exist_ok=True)

results = yolo.predict(
    source=picks,
    imgsz=960,
    conf=0.20,
    iou=0.50,
    device=0,
    verbose=False
)

for i, r in enumerate(results, 1):
    vis = r.plot(boxes=False)  # BGR image with mask overlay only
    out_path = YOLO_OUT / f"{i:02d}__{Path(r.path).name}"
    cv2.imwrite(str(out_path), vis)

print("Saved YOLO results to:", YOLO_OUT.resolve())


Saved YOLO results to: E:\dl_cv_mini_project\road_lane_detection\outputs\yolo_predictions_same20_no_boxes


In [38]:
import shutil
from pathlib import Path

CMP = Path("outputs/compare_unet_vs_yolo")
ORIG = CMP / "original"
UNET = CMP / "unet_hough"
YOLO_DIR = CMP / "yolo"

for d in [ORIG, UNET, YOLO_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Copy originals
for i, p in enumerate([Path(x) for x in picks], 1):
    if p.exists():
        shutil.copy2(p, ORIG / f"{i:02d}__{p.name}")

# Copy UNet+Hough outputs (after)
unet_after_dir = Path("outputs/random_test_results_after")
unet_files = list(unet_after_dir.glob("*.jpg"))

for i, p in enumerate([Path(x) for x in picks], 1):
    fname = p.name
    cand = [u for u in unet_files if u.name.endswith(fname)]
    if cand:
        shutil.copy2(cand[0], UNET / f"{i:02d}__{fname}")
    else:
        print("[WARN] UNet output not found for:", fname)

# Copy YOLO no-box outputs
for i, p in enumerate([Path(x) for x in picks], 1):
    fname = p.name
    yolo_file = YOLO_OUT / f"{i:02d}__{fname}"
    if yolo_file.exists():
        shutil.copy2(yolo_file, YOLO_DIR / yolo_file.name)
    else:
        print("[WARN] YOLO output not found for:", fname)

print("Comparison folder ready:", CMP.resolve())


Comparison folder ready: E:\dl_cv_mini_project\road_lane_detection\outputs\compare_unet_vs_yolo
